In [0]:
# 1. Read the raw Parquet data from Bronze layer
# No config needed because of the External Location we just set up
bronze_path = "gs://skyline-landing-zone-marmaris/bronze/*.parquet"
df_bronze = spark.read.parquet(bronze_path)

# 2. Transformation Logic (Silver Layer)
from pyspark.sql.functions import col, when, current_timestamp

# - Removing rows without CustomerId
# - Dropping duplicates to ensure data quality
# - Adding age_group for business insights
# - Adding a processing timestamp
df_silver = df_bronze \
    .filter(col("CustomerId").isNotNull()) \
    .dropDuplicates(["CustomerId"]) \
    .withColumn("age_group", 
        when(col("Age") < 30, "Young")
        .when((col("Age") >= 30) & (col("Age") < 50), "Middle-Aged")
        .otherwise("Senior")) \
    .withColumn("processed_at", current_timestamp())

# 3. Write the cleaned data to Silver layer in Delta format
# Delta format allows features like Time Travel and ACID transactions
silver_output_path = "gs://skyline-landing-zone-marmaris/silver/bank_churn_silver"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_output_path)

print(f"Silver Layer successfully created at: {silver_output_path}")
display(df_silver.limit(10))

Silver Layer successfully created at: gs://skyline-landing-zone-marmaris/silver/bank_churn_silver


RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ingested_at,age_group,processed_at
28,15700772,Nebechi,571,France,Male,44,9,0.0,2,0,0,38433.35,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
31,15589475,Azikiwe,591,Spain,Female,39,3,0.0,3,1,0,140469.38,1,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
63,15702014,Jeffrey,555,Spain,Male,33,1,56084.69,2,0,0,178798.13,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
66,15789484,Hammond,751,Germany,Female,36,6,169831.46,2,1,1,27758.36,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
96,15699461,Fiorentini,515,Spain,Male,35,10,176273.95,1,0,1,121277.78,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
99,15604348,Allard,710,Spain,Male,22,8,0.0,2,0,0,99645.04,0,2026-04-25 21:22:55,Young,2026-04-25T20:31:49.587Z
157,15655007,Li,758,France,Female,33,7,0.0,2,0,0,82996.47,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
184,15810845,T'ang,636,France,Male,42,2,0.0,2,1,1,55470.78,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
204,15727868,Onuora,711,France,Female,38,2,129022.06,2,1,1,14374.86,1,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z
210,15612087,Dike,671,France,Male,45,2,106376.85,1,0,1,158264.62,0,2026-04-25 21:22:55,Middle-Aged,2026-04-25T20:31:49.587Z


In [0]:
from pyspark.sql.functions import avg, count

# Aggregate data to find churn rate per age group
df_gold_age_group = df_silver.groupBy("age_group") \
    .agg(
        count("CustomerId").alias("total_customers"),
        avg("Exited").alias("churn_rate")
    ) \
    .orderBy("churn_rate", ascending=False)

# Save the Gold table (This will be our source for BigQuery and Streamlit)
gold_output_path = "gs://skyline-landing-zone-marmaris/gold/churn_by_age_group"

df_gold_age_group.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_output_path)

print("Gold Layer (Business Insights) is ready!")
display(df_gold_age_group)

Gold Layer (Business Insights) is ready!


age_group,total_customers,churn_rate
Senior,1395,0.4544802867383513
Middle-Aged,6964,0.18365881677197013
Young,1641,0.07556368068251067
